# 05 — Graph Feature Analysis

Explores the entity graph structure and shows how graph-derived features (degree, PageRank, neighbour fraud ratio, community fraud ratio) capture linked-entity risk.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.feature_store import build_features
from src.models.graph_features import build_graph_features

## Load and inspect the edge table

In [ ]:
edges = pd.read_csv("../data/synthetic/edges.csv")
print(f"Total edges: {len(edges):,}")
print(f"\nEdge types:")
print(edges["edge_type"].value_counts())
print(f"\nUnique source entities: {edges['src_entity_id'].nunique()}")
print(f"Unique destination entities: {edges['dst_entity_id'].nunique()}")

## Build graph features on training data

In [ ]:
train = pd.read_csv("../data/synthetic/train.csv")
accounts = pd.read_csv("../data/synthetic/accounts.csv")
customers = pd.read_csv("../data/synthetic/customers.csv")

df = build_features(train, accounts, customers)
df = build_graph_features(df)

graph_cols = ["graph_degree", "graph_neighbor_fraud_ratio", "graph_pagerank", "graph_community_fraud_ratio"]
print(df[graph_cols].describe())

## Graph features by fraud label

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, graph_cols):
    fraud_vals = df.loc[df["label_fraud"] == 1, col]
    legit_vals = df.loc[df["label_fraud"] == 0, col]
    ax.hist(legit_vals.clip(upper=legit_vals.quantile(0.99)), bins=30, alpha=0.6, label="Non-fraud")
    ax.hist(fraud_vals.clip(upper=fraud_vals.quantile(0.99)), bins=30, alpha=0.6, label="Fraud")
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

## Correlation of graph features with fraud

In [ ]:
for col in graph_cols:
    fraud_mean = df.loc[df["label_fraud"] == 1, col].mean()
    legit_mean = df.loc[df["label_fraud"] == 0, col].mean()
    corr = df[col].corr(df["label_fraud"])
    print(f"{col:35s}  fraud_mean={fraud_mean:.4f}  legit_mean={legit_mean:.4f}  corr={corr:.4f}")

## Key findings

- Graph features capture linked-entity risk that pure transaction features miss.
- Neighbour fraud ratio directly measures exposure to known-fraud accounts in the entity network.
- Community fraud ratio identifies clusters of accounts with elevated collective risk.
- These features would be even more powerful with richer data (shared addresses, phone numbers, IP ranges).